In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np

from src.features import compute_hedge_ratio, compute_kalman_hedge, compute_calendar_features, compute_spread_vol
from src.labels import get_daily_vol, get_vertical_barrier, apply_pt_sl_on_t1, get_bins, get_avg_uniqueness, seq_bootstrap
from src.cv import temporal_split, WalkForwardPurgedCV, cv_score, feat_importance_mda, feat_importance_mdi

## 1. Features

Compute all features on the full dataset before splitting.

In [2]:
df = pd.read_csv("../data/full_dataset.csv", index_col="date", parse_dates=True)
df = compute_hedge_ratio(df, 'close_corn', 'close_soybean')         # OLS (504-day window)
df = compute_kalman_hedge(df, 'close_corn', 'close_soybean')        # Kalman 3-state
df = compute_calendar_features(df)                                   # month, day_of_week
df = compute_spread_vol(df, 'close_corn', 'close_soybean', 'kf_hedge_ratio')  # EWMA vol

Window:          504 trading days
Valid rows:      3148 / 3652
Hedge ratio:     -0.0992 to 1.3937
Intercept:       -5.8013 to 23.9495
Spread mean:     0.0989
Converged in 8 iteration(s)
  phi      = 0.996003  (half-life = 250 trading days)
  sigma2_s = 0.023109
  delta    = 1.0e-05  (hyperparameter — tune via CV)
  R        = 1.0e-06  (numerical floor)
Hedge ratio:      0.2550 to 0.7351
Spread level std: 1.7017
Innovation z std: 1.1113 (ideal ~ 1.0)
Level z std:      1.2242


## 2. Labels

Triple barrier labeling (AFML Ch.3) + sample uniqueness weights (Ch.4).
Restrict to dates where OLS hedge ratio is valid (after 504-day burn-in).

In [39]:
corn        = df['close_corn']
soy         = df['close_soybean']
hedge_ratio = df['hedge_ratio']

# Only label dates where the OLS hedge ratio exists
valid_dates = df.dropna(subset=['hedge_ratio']).index

vol     = get_daily_vol(corn, soy, hedge_ratio)
t1      = get_vertical_barrier(valid_dates, num_days=150, t_events=valid_dates)
touches = apply_pt_sl_on_t1(corn, soy, hedge_ratio, t1, vol, pt_sl=[2, 2])
labels  = get_bins(touches, corn, soy, hedge_ratio)
weights = get_avg_uniqueness(labels, df.index)

print(f"\nLabeled events: {len(labels)}")
print(f"Label distribution: {labels['bin'].value_counts().to_dict()}")
print(f"Mean uniqueness: {weights.mean():.4f}")


Labeled events: 2998
Label distribution: {1: 1571, -1: 1427}
Mean uniqueness: 0.1187


## 3. Assemble X, y, t1, w

Build the full feature matrix: 4 Kalman-derived + 2 calendar + 1 spread vol + 90 weather = 97 features.
Exclude raw prices, volumes, and intermediate estimation columns (hedge_ratio, intercept, spread)
which would leak information or are not meaningful as predictors.

In [52]:
# Columns to EXCLUDE from features
exclude_cols = [
    'close_corn', 'close_soybean', 'high_corn', 'high_soybean',
    'low_corn', 'low_soybean', 'open_corn', 'open_soybean',
    'volume_corn', 'volume_soybean',
    'hedge_ratio', 'intercept',#'spread',       # OLS intermediates
    'kf_hedge_ratio', 'kf_intercept',            # Kalman intermediates
]

feature_cols = [c for c in df.columns if c not in exclude_cols]

X         = df.loc[labels.index, feature_cols].copy()
y         = labels['bin'].map({-1: 0, 1: 1})   # XGBoost needs {0, 1}
t1_series = labels['t1']
w         = weights

# Sanity check for NaNs
nan_count = X.isna().sum()
if nan_count.any():
    print(f"WARNING: NaN features found:")
    print(nan_count[nan_count > 0])
else:
    print(f"No NaN features — all {X.shape[0]} rows clean")

print(f"\nFeatures: {len(feature_cols)}")
print(f"  Weather:  {len([c for c in feature_cols if 'roll30d' in c])}")
print(f"  Kalman:   {[c for c in feature_cols if c.startswith('kf_')]}")
print(f"  Other:    {[c for c in feature_cols if 'roll30d' not in c and not c.startswith('kf_')]}")
print(f"\nShape: {X.shape}")
print(f"Label dist: {y.value_counts().sort_index().to_dict()}")

No NaN features — all 2998 rows clean

Features: 98
  Weather:  90
  Kalman:   ['kf_spread', 'kf_innovation', 'kf_z_score', 'kf_level_z']
  Other:    ['spread', 'month', 'day_of_week', 'spread_vol']

Shape: (2998, 98)
Label dist: {0: 1427, 1: 1571}


## 4. Dev / Holdout Split

Reserve the last 1.5 years (375 trading days) as a final holdout set.
All model development happens on the dev set only. Holdout is touched once at the end.

In [53]:
split = temporal_split(X, y, t1_series, sample_weight=w, n_holdout=375)

Dev set:      2623 obs  (2013-09-20 → 2024-02-22)
Holdout set:   375 obs  (2024-02-23 → 2025-08-21)
Cutoff date: 2024-02-23
Dev label dist:     {0: 1252, 1: 1371}
Holdout label dist: {0: 175, 1: 200}


## 5. Baseline: 6-Feature XGBoost

Kalman + calendar + spread vol only. Default hyperparameters.
Establishes the baseline to beat.

In [54]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=42,
)

baseline_cols = ['kf_level_z', 'kf_z_score', 'kf_spread', 'spread_vol', 'month', 'day_of_week']
split_base = temporal_split(X[baseline_cols], y, t1_series, sample_weight=w, n_holdout=375)

cv_base = WalkForwardPurgedCV(n_periods=3, t1=split_base['t1_dev'])
print("\n=== Baseline (6 features) — Accuracy ===")
res_base = cv_score(model, split_base['X_dev'], split_base['y_dev'], split_base['t1_dev'],
                    sample_weight=split_base['w_dev'], cv=cv_base)

Dev set:      2623 obs  (2013-09-20 → 2024-02-22)
Holdout set:   375 obs  (2024-02-23 → 2025-08-21)
Cutoff date: 2024-02-23
Dev label dist:     {0: 1252, 1: 1371}
Holdout label dist: {0: 175, 1: 200}

=== Baseline (6 features) — Accuracy ===
Fold    Train   Test  Purged               Train Dates                Test Dates  Train Score  Test Score
---------------------------------------------------------------------------------------------------------
1         870    874       4   2013-09-20 → 2017-03-06   2017-03-13 → 2020-08-28       0.8034      0.5092
2        1746    875       2   2013-09-20 → 2020-08-26   2020-08-31 → 2024-02-22       0.7795      0.4571
---------------------------------------------------------------------------------------------------------
Mean                                                                                   0.7915      0.4831
Std                                                                                    0.0120      0.0260


## 6. Full Feature Set XGBoost (97 features)

All weather + Kalman + calendar + spread vol. Same default hyperparameters.
Expect more overfitting but potentially better test performance if weather carries signal.

In [55]:
cv_full = WalkForwardPurgedCV(n_periods=3, t1=split['t1_dev'])

print("=== Full Feature Set (97 features) — Accuracy ===")
res_full = cv_score(model, split['X_dev'], split['y_dev'], split['t1_dev'],
                    sample_weight=split['w_dev'], cv=cv_full)

print("\n=== Full Feature Set — Neg Log Loss ===")
cv_full_ll = WalkForwardPurgedCV(n_periods=3, t1=split['t1_dev'])
res_full_ll = cv_score(model, split['X_dev'], split['y_dev'], split['t1_dev'],
                       sample_weight=split['w_dev'], cv=cv_full_ll, scoring='neg_log_loss')

=== Full Feature Set (97 features) — Accuracy ===
Fold    Train   Test  Purged               Train Dates                Test Dates  Train Score  Test Score
---------------------------------------------------------------------------------------------------------
1         870    874       4   2013-09-20 → 2017-03-06   2017-03-13 → 2020-08-28       0.9345      0.5080
2        1746    875       2   2013-09-20 → 2020-08-26   2020-08-31 → 2024-02-22       0.9101      0.5371
---------------------------------------------------------------------------------------------------------
Mean                                                                                   0.9223      0.5226
Std                                                                                    0.0122      0.0146

=== Full Feature Set — Neg Log Loss ===
Fold    Train   Test  Purged               Train Dates                Test Dates  Train Score  Test Score
-------------------------------------------------------------

## 7. MDA Feature Importance (AFML Ch.8)

Permutation importance using walk-forward CV. For each feature, shuffle it and
measure the drop in OOS performance. Positive = useful, negative = detrimental.

In [56]:
cv_mda = WalkForwardPurgedCV(n_periods=3, t1=split['t1_dev'])
mda = feat_importance_mda(model, split['X_dev'], split['y_dev'], split['t1_dev'],
                          sample_weight=split['w_dev'], cv=cv_mda)

print("Top 20 features:")
print(mda.head(20).to_string())
print(f"\nBottom 10:")
print(mda.tail(10).to_string())
print(f"\nPositive importance: {(mda['mean'] > 0).sum()} / {len(mda)}")

Top 20 features:
                                                                mean       std
wind_speed_10m_max_parana_roll30d_max                       0.018637  0.000055
precipitation_sum_central_indiana_roll30d_sum               0.013957  0.011289
soil_moisture_0_to_7cm_mean_parana_roll30d_min              0.007319  0.003737
wind_speed_10m_max_central_indiana_roll30d_max              0.004970  0.003773
precipitation_sum_rio_grande_sul_roll30d_sum                0.004889  0.000476
wind_speed_10m_max_rio_grande_sul_roll30d_max               0.004568  0.003268
relative_humidity_2m_mean_central_indiana_roll30d_mean      0.004318  0.001333
precipitation_sum_central_illinois_roll30d_sum              0.004167  0.005653
relative_humidity_2m_mean_rio_grande_sul_roll30d_mean       0.003592  0.006985
soil_moisture_0_to_7cm_mean_central_illinois_roll30d_min    0.003320  0.000598
soil_moisture_0_to_100cm_mean_central_illinois_roll30d_min  0.002798  0.002798
precipitation_sum_central_iowa_roll

## 8. MDI Feature Importance

In [57]:

mdi = feat_importance_mdi(split['X_dev'], split['y_dev'], sample_weight=split['w_dev'])

print("MDI Top 20:")
print(mdi.head(20).to_string())

MDI Top 20:
                                                                 mean       std
kf_level_z                                                   0.014195  0.000481
kf_spread                                                    0.014056  0.000436
spread                                                       0.013802  0.000426
spread_vol                                                   0.013302  0.000426
precipitation_sum_central_indiana_roll30d_sum                0.012637  0.000422
temperature_2m_min_mato_grosso_roll30d_mean                  0.012532  0.000416
wet_bulb_temperature_2m_mean_mato_grosso_roll30d_mean        0.012275  0.000415
soil_temperature_0_to_7cm_mean_mato_grosso_roll30d_mean      0.011867  0.000382
relative_humidity_2m_mean_rio_grande_sul_roll30d_mean        0.011845  0.000363
precipitation_sum_central_illinois_roll30d_sum               0.011821  0.000405
relative_humidity_2m_mean_central_indiana_roll30d_mean       0.011800  0.000408
vapour_pressure_deficit_max_

## 9. Union of MDI & MDA

In [58]:
mdi_top20 = set(mdi.head(20).index)
mda_top20 = set(mda.head(20).index)
union = mdi_top20 | mda_top20

vetoed = [f for f in union if mda.loc[f, 'mean'] < 0]
selected = sorted([f for f in union if mda.loc[f, 'mean'] >= 0])

print(f"MDI top 20:        {len(mdi_top20)}")
print(f"MDA top 20:        {len(mda_top20)}")
print(f"Union:             {len(union)}")
print(f"Vetoed (MDA < 0):  {len(vetoed)}")
print(f"Final selection:   {len(selected)}")

print(f"\nVetoed features:")
for f in vetoed:
    print(f"  {f}: MDI rank {list(mdi.index).index(f)+1}, MDA = {mda.loc[f,'mean']:.4f}")

print(f"\nSelected features ({len(selected)}):")
print(f"{'Feature':<65} {'MDI rank':>9} {'MDA rank':>9} {'MDA mean':>9}")
print("-" * 95)
for f in selected:
    mdi_rank = list(mdi.index).index(f) + 1
    mda_rank = list(mda.index).index(f) + 1
    print(f"{f:<65} {mdi_rank:>9} {mda_rank:>9} {mda.loc[f,'mean']:>9.4f}")

MDI top 20:        20
MDA top 20:        20
Union:             34
Vetoed (MDA < 0):  10
Final selection:   24

Vetoed features:
  vapour_pressure_deficit_max_central_iowa_roll30d_mean: MDI rank 20, MDA = -0.0013
  spread: MDI rank 3, MDA = -0.0123
  temperature_2m_min_mato_grosso_roll30d_mean: MDI rank 6, MDA = -0.0000
  spread_vol: MDI rank 4, MDA = -0.0024
  et0_fao_evapotranspiration_sum_central_illinois_roll30d_sum: MDI rank 19, MDA = -0.0011
  kf_spread: MDI rank 2, MDA = -0.0008
  vapour_pressure_deficit_max_central_indiana_roll30d_mean: MDI rank 15, MDA = -0.0021
  vapour_pressure_deficit_max_parana_roll30d_mean: MDI rank 12, MDA = -0.0009
  soil_temperature_0_to_7cm_mean_parana_roll30d_mean: MDI rank 13, MDA = -0.0039
  wet_bulb_temperature_2m_mean_mato_grosso_roll30d_mean: MDI rank 7, MDA = -0.0009

Selected features (24):
Feature                                                            MDI rank  MDA rank  MDA mean
------------------------------------------------------------

## 10. Retrain XGBoost with Selected Features


In [59]:
# Cell 10: Retrain with selected features (uses `selected` from cell 9)
print(f"Selected features: {len(selected)}\n")

split_sel = temporal_split(X[selected], y, t1_series, sample_weight=w, n_holdout=375)

cv_sel = WalkForwardPurgedCV(n_periods=3, t1=split_sel['t1_dev'])
print("\n=== XGBoost with selected features ===")
res_sel = cv_score(model, split_sel['X_dev'], split_sel['y_dev'], split_sel['t1_dev'],
                   sample_weight=split_sel['w_dev'], cv=cv_sel)

Selected features: 24

Dev set:      2623 obs  (2013-09-20 → 2024-02-22)
Holdout set:   375 obs  (2024-02-23 → 2025-08-21)
Cutoff date: 2024-02-23
Dev label dist:     {0: 1252, 1: 1371}
Holdout label dist: {0: 175, 1: 200}

=== XGBoost with selected features ===
Fold    Train   Test  Purged               Train Dates                Test Dates  Train Score  Test Score
---------------------------------------------------------------------------------------------------------
1         870    874       4   2013-09-20 → 2017-03-06   2017-03-13 → 2020-08-28       0.9069      0.5229
2        1746    875       2   2013-09-20 → 2020-08-26   2020-08-31 → 2024-02-22       0.8603      0.6343
---------------------------------------------------------------------------------------------------------
Mean                                                                                   0.8836      0.5786
Std                                                                                    0.0233      0.

## 11. Random Forest with Sequential Bootstrap (AFML Ch.4.5)

RF requires sequential bootstrap because standard bootstrap oversamples redundant
observations (mean uniqueness ≈ 0.12). We draw bags proportional to each observation's
uniqueness, then train individual decision trees and average predictions.

In [60]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, log_loss
from sklearn.base import clone

n_trees = 200

# Evaluate on each walk-forward fold
cv_rf = WalkForwardPurgedCV(n_periods=3, t1=split_sel['t1_dev'])

for fold_i, (train_idx, test_idx) in enumerate(cv_rf.split(split_sel['X_dev'])):
    X_train = split_sel['X_dev'].iloc[train_idx]
    y_train = split_sel['y_dev'].iloc[train_idx]
    w_train = split_sel['w_dev'].iloc[train_idx]
    X_test  = split_sel['X_dev'].iloc[test_idx]
    y_test  = split_sel['y_dev'].iloc[test_idx]
    
    t1_train = split_sel['t1_dev'].iloc[train_idx]
    
    avg_u = w_train.mean()
    s_length = max(int(avg_u * len(t1_train)), 50)
    
    print(f"Fold {fold_i+1}: train={len(X_train)}, s_length={s_length}")
    
    # Train trees on sequential bootstrap bags
    trees = []
    for tree_i in range(n_trees):
        bag_idx = seq_bootstrap(t1_train, df.index, s_length=s_length,
                               random_state=tree_i)
        
        dt = DecisionTreeClassifier(
            criterion='entropy',
            max_features='sqrt',
            max_depth=5,
            min_samples_leaf=max(1, int(0.05 * s_length)),
            class_weight='balanced',
            random_state=tree_i,
        )
        dt.fit(X_train.iloc[bag_idx].values, y_train.iloc[bag_idx].values,
               sample_weight=w_train.iloc[bag_idx].values)
        trees.append(dt)
    
    # Predict: average probabilities
    probs = np.array([t.predict_proba(X_test.values) for t in trees])
    avg_prob = probs.mean(axis=0)
    pred = (avg_prob[:, 1] >= 0.5).astype(int)
    
    acc = accuracy_score(y_test.values, pred)
    ll = log_loss(y_test.values, avg_prob)
    print(f"  Seq Bootstrap RF:  acc={acc:.4f}, neg_ll={-ll:.4f}")
    
    # Compare with XGBoost on same fold
    m = clone(model)
    m.fit(X_train.values, y_train.values, sample_weight=w_train.values)
    xgb_acc = accuracy_score(y_test.values, m.predict(X_test.values))
    xgb_ll = log_loss(y_test.values, m.predict_proba(X_test.values))
    print(f"  XGBoost:           acc={xgb_acc:.4f}, neg_ll={-xgb_ll:.4f}\n")

Fold 1: train=870, s_length=97


KeyboardInterrupt: 

## 12. XGBoost Hyperparameter Tuning

In [61]:
from itertools import product
from sklearn.base import clone
from sklearn.metrics import accuracy_score, log_loss

param_grid = list(product(
    [2, 3, 4],           # max_depth
    [0.01, 0.05, 0.1],   # learning_rate
    [100, 200, 300],      # n_estimators
    [1, 10, 30],          # min_child_weight
))

print(f"Testing {len(param_grid)} XGBoost configurations...\n")

xgb_results = []
for md, lr, ne, mcw in param_grid:
    m = XGBClassifier(max_depth=md, learning_rate=lr, n_estimators=ne,
                      min_child_weight=mcw, eval_metric='logloss', random_state=42)
    
    cv = WalkForwardPurgedCV(n_periods=3, t1=split_sel['t1_dev'])
    train_sc, test_sc = [], []
    for train_idx, test_idx in cv.split(split_sel['X_dev']):
        m_clone = clone(m)
        m_clone.fit(split_sel['X_dev'].iloc[train_idx].values,
                    split_sel['y_dev'].iloc[train_idx].values,
                    sample_weight=split_sel['w_dev'].iloc[train_idx].values)
        train_sc.append(-log_loss(split_sel['y_dev'].iloc[train_idx].values,
                                       m_clone.predict(split_sel['X_dev'].iloc[train_idx].values)))
        test_sc.append(-log_loss(split_sel['y_dev'].iloc[test_idx].values,
                                      m_clone.predict(split_sel['X_dev'].iloc[test_idx].values)))
    
    xgb_results.append({'max_depth': md, 'learning_rate': lr, 'n_estimators': ne,
                        'min_child_weight': mcw,
                        'train_acc': np.mean(train_sc), 'test_acc': np.mean(test_sc),
                        'gap': np.mean(train_sc) - np.mean(test_sc)})

xgb_df = pd.DataFrame(xgb_results).sort_values('test_acc', ascending=False)
print("Top 15 by test accuracy:")
print(xgb_df.head(15).to_string(index=False))

Testing 81 XGBoost configurations...

Top 15 by test accuracy:
 max_depth  learning_rate  n_estimators  min_child_weight  train_acc   test_acc       gap
         4           0.05           300                 1  -2.656328 -14.674115 12.017787
         2           0.05           300                 1  -5.675199 -14.756382  9.081183
         4           0.05           200                 1  -3.276417 -15.003704 11.727286
         2           0.10           200                 1  -4.537739 -15.024206 10.486467
         3           0.10           300                 1  -2.232636 -15.024489 12.791852
         4           0.05           100                 1  -4.858355 -15.106615 10.248260
         3           0.05           300                 1  -3.317918 -15.127447 11.809529
         4           0.10           200                 1  -2.098097 -15.147902 13.049805
         2           0.10           300                 1  -3.534747 -15.189001 11.654254
         3           0.10           1

## 13. Pipeline Hyperparameter Tuning

In [62]:
# === CELL: Pipeline Hyperparameter Tuning ===
# Uses best XGBoost params from above. Takes ~15-30 min on your MacBook.

best_model = XGBClassifier(
    max_depth=int(xgb_df.iloc[0]['max_depth']),
    learning_rate=xgb_df.iloc[0]['learning_rate'],
    n_estimators=int(xgb_df.iloc[0]['n_estimators']),
    min_child_weight=int(xgb_df.iloc[0]['min_child_weight']),
    eval_metric='logloss', random_state=42,
)

delta_grid = [1e-6, 1e-5, 1e-4]
num_days_grid = [100, 150, 200]
pt_sl_grid = [1.5, 2.0, 2.5]

pipe_results = []
total = len(delta_grid) * len(num_days_grid) * len(pt_sl_grid)
combo = 0

for delta in delta_grid:
    # Re-run Kalman with this delta
    df_temp = df.drop(columns=[c for c in df.columns if c.startswith('kf_')], errors='ignore')
    df_temp = compute_kalman_hedge(df_temp, 'close_corn', 'close_soybean', delta=delta)
    
    for num_days in num_days_grid:
        for pt_sl_mult in pt_sl_grid:
            combo += 1
            
            # Re-run labeling
            vol_t = get_daily_vol(corn, soy, hedge_ratio)
            t1_t = get_vertical_barrier(valid_dates, num_days=num_days, t_events=valid_dates)
            touches_t = apply_pt_sl_on_t1(corn, soy, hedge_ratio, t1_t, vol_t,
                                          pt_sl=[pt_sl_mult, pt_sl_mult])
            labels_t = get_bins(touches_t, corn, soy, hedge_ratio)
            weights_t = get_avg_uniqueness(labels_t, df.index)
            
            X_t = df_temp.loc[labels_t.index, selected].copy()
            y_t = labels_t['bin'].map({-1: 0, 1: 1})
            t1_t_s, w_t = labels_t['t1'], weights_t
            
            sp = temporal_split(X_t, y_t, t1_t_s, sample_weight=w_t, n_holdout=375)
            
            cv = WalkForwardPurgedCV(n_periods=3, t1=sp['t1_dev'])
            train_sc, test_sc = [], []
            for train_idx, test_idx in cv.split(sp['X_dev']):
                m = clone(best_model)
                m.fit(sp['X_dev'].iloc[train_idx].values, sp['y_dev'].iloc[train_idx].values,
                      sample_weight=sp['w_dev'].iloc[train_idx].values)
                train_sc.append(-log_loss(sp['y_dev'].iloc[train_idx].values,
                                               m.predict(sp['X_dev'].iloc[train_idx].values)))
                test_sc.append(-log_loss(sp['y_dev'].iloc[test_idx].values,
                                              m.predict(sp['X_dev'].iloc[test_idx].values)))
            
            pipe_results.append({
                'delta': delta, 'num_days': num_days, 'pt_sl': pt_sl_mult,
                'n_events': len(labels_t),
                'train_acc': np.mean(train_sc), 'test_acc': np.mean(test_sc),
                'gap': np.mean(train_sc) - np.mean(test_sc),
            })
            
            print(f"[{combo}/{total}] delta={delta:.0e} days={num_days} pt_sl={pt_sl_mult} "
                  f"events={len(labels_t)} train={np.mean(train_sc):.3f} test={np.mean(test_sc):.3f}")

pipe_df = pd.DataFrame(pipe_results).sort_values('test_acc', ascending=False)
print("\nTop 10 pipeline configurations:")
print(pipe_df.head(10).to_string(index=False))

Converged in 4 iteration(s)
  phi      = 0.994075  (half-life = 168 trading days)
  sigma2_s = 0.030082
  delta    = 1.0e-06  (hyperparameter — tune via CV)
  R        = 1.0e-06  (numerical floor)
Hedge ratio:      0.2787 to 0.6499
Spread level std: 1.5955
Innovation z std: 1.1045 (ideal ~ 1.0)
Level z std:      1.1798
Dev set:      2673 obs  (2013-09-20 → 2024-05-03)
Holdout set:   375 obs  (2024-05-06 → 2025-10-31)
Cutoff date: 2024-05-06
Dev label dist:     {0: 1285, 1: 1388}
Holdout label dist: {0: 175, 1: 200}
[1/27] delta=1e-06 days=100 pt_sl=1.5 events=3048 train=-2.221 test=-15.514
Dev set:      2673 obs  (2013-09-20 → 2024-05-03)
Holdout set:   375 obs  (2024-05-06 → 2025-10-31)
Cutoff date: 2024-05-06
Dev label dist:     {0: 1273, 1: 1400}
Holdout label dist: {0: 172, 1: 203}
[2/27] delta=1e-06 days=100 pt_sl=2.0 events=3048 train=-2.243 test=-15.129
Dev set:      2673 obs  (2013-09-20 → 2024-05-03)
Holdout set:   375 obs  (2024-05-06 → 2025-10-31)
Cutoff date: 2024-05-06
Dev